In [46]:
# 📰 Quiz 1 – NLP News Pipeline
# 1 - Scrape technology news headlines & summaries from a website
# 2 - Clean the text (remove digits, punctuation, extra spaces, ...)
# 3 - Tokenize the text
# 4 - Remove stopwords
# 5 - Do POS tagging
# 6 - Perform Named Entity Recognition (NER)
# 7 - Extract all NOUN tokens and PERSON entities
# 8 - Print the most frequent nouns and person

In [47]:
# 1
import requests as rq
from bs4 import BeautifulSoup

url = "https://edition.cnn.com/business/tech"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/118.0.0.0 Safari/537.36"
    )
}

try:
    res = rq.get(url, headers=headers, timeout=10)
    res.raise_for_status()
except rq.exceptions.HTTPError as e:
    print("❌ HTTP Error:", e)
    res = None
except rq.exceptions.ConnectionError:
    print("❌ Connection Error: Check your internet!")
    res = None
except rq.exceptions.Timeout:
    print("❌ Timeout Error!")
    res = None
except Exception as e:
    print("❌ Unexpected Error:", e)
    res = None
else:
    print("Page fetched successfully!✅ ")

if res is None:
    raise SystemExit("Request failed, cannot continue.")

res.encoding = "utf-8"
soup = BeautifulSoup(res.text, "lxml")

titles = []
summaries = []

for block in soup.find_all("div", class_="container__headline"):
    title_tag = block.find("span", class_="container__headline-text")
    if title_tag:
        titles.append(title_tag.get_text())

for p in soup.find_all("p"):
    summaries.append(p.get_text())


print("number of titles:", len(titles))
print("number of summaries:", len(summaries))

if titles:
    print("Sample title:", titles[0])
if summaries:
    print("Sample summary:", summaries[0])

raw_text = " . ".join(titles + summaries)
print("Raw text (first 400 chars):")
print(raw_text[:400])

Page fetched successfully!✅ 
number of titles: 27
number of summaries: 7
Sample title: What on Earth just happened to the stock market?
Sample summary: Markets 



Raw text (first 400 chars):
What on Earth just happened to the stock market? . The biggest reason America is turning on Target and switching to Walmart . The air traffic problem making your holiday travel even more miserable . Why the world’s richest man and the CEO of the most valuable company met with Saudi officials . Trump renews effort to block states from regulating AI, raising alarms about safety . AI teddy bear pulle


In [48]:
# 2
import re
def clean_text(text: str) -> str:
    """Clean the text (remove digits, punctuation, extra spaces, ...) """
    text = text.lower()
    text = re.sub(r"\d+", " ", text)  #remove digits
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)  # collapse spaces/newlines/tabs to 1 space
    text = text.strip()
    return text

cleaned_text = clean_text(raw_text)

print("Cleaned text (first 400 chars):")
print(cleaned_text[:400])

Cleaned text (first 400 chars):
what on earth just happened to the stock market the biggest reason america is turning on target and switching to walmart the air traffic problem making your holiday travel even more miserable why the world s richest man and the ceo of the most valuable company met with saudi officials trump renews effort to block states from regulating ai raising alarms about safety ai teddy bear pulled after givi


In [49]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 56.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [50]:
# 3
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp(cleaned_text)

tokens = [token.text for token in doc if not token.is_space]

print("total tokens:", len(tokens))
print("Sample 20 tokens:", tokens[:20])


total tokens: 520
Sample 20 tokens: ['what', 'on', 'earth', 'just', 'happened', 'to', 'the', 'stock', 'market', 'the', 'biggest', 'reason', 'america', 'is', 'turning', 'on', 'target', 'and', 'switching', 'to']


In [51]:
# 4 -  Stopwords
content_tokens = [
    token for token in doc
    if not token.is_space and not token.is_punct and not token.is_stop
]

print("tokens without stopwords:", len(content_tokens))
print("Sample 20 content tokens:", [t.text for t in content_tokens[:20]])

tokens without stopwords: 342
Sample 20 content tokens: ['earth', 'happened', 'stock', 'market', 'biggest', 'reason', 'america', 'turning', 'target', 'switching', 'walmart', 'air', 'traffic', 'problem', 'making', 'holiday', 'travel', 'miserable', 'world', 's']


In [52]:
# 5
print("POS samples (first 40 tokens):")
for token in content_tokens[:40]:
    print(f"{token.text:15}  POS={token.pos_:10}  TAG={token.tag_}")


POS samples (first 40 tokens):
earth            POS=NOUN        TAG=NN
happened         POS=VERB        TAG=VBD
stock            POS=NOUN        TAG=NN
market           POS=NOUN        TAG=NN
biggest          POS=ADJ         TAG=JJS
reason           POS=NOUN        TAG=NN
america          POS=PROPN       TAG=NNP
turning          POS=VERB        TAG=VBG
target           POS=NOUN        TAG=NN
switching        POS=VERB        TAG=VBG
walmart          POS=VERB        TAG=VB
air              POS=NOUN        TAG=NN
traffic          POS=NOUN        TAG=NN
problem          POS=NOUN        TAG=NN
making           POS=VERB        TAG=VBG
holiday          POS=NOUN        TAG=NN
travel           POS=NOUN        TAG=NN
miserable        POS=ADJ         TAG=JJ
world            POS=NOUN        TAG=NN
s                POS=PART        TAG=POS
richest          POS=ADJ         TAG=JJS
man              POS=NOUN        TAG=NN
ceo              POS=NOUN        TAG=NN
valuable         POS=ADJ         TAG=JJ
c

In [53]:
# 6
print(doc.ents)
entities = [(ent.text, ent.label_) for ent in doc.ents]
print("\nNamed Entities (first 30):")
for text, label in entities[:30]:
    print(f"{text:30} -> {label}")

(america, the world s richest, saudi, america, year old, decades, nasa, two decades, america, september, every two minutes, factset research systems inc, chicago, chicago mercantile exchange inc, s p dow jones indices llc, cnn standard, copp clark limited cable, warner bros discovery, cnn, sans cable)

Named Entities (first 30):
america                        -> GPE
the world s richest            -> ORG
saudi                          -> NORP
america                        -> GPE
year old                       -> DATE
decades                        -> DATE
nasa                           -> PERSON
two decades                    -> DATE
america                        -> GPE
september                      -> DATE
every two minutes              -> TIME
factset research systems inc   -> ORG
chicago                        -> GPE
chicago mercantile exchange inc -> ORG
s p dow jones indices llc      -> ORG
cnn standard                   -> ORG
copp clark limited cable       -> PERSON
warner bro

In [54]:
# 7
#Extract all NOUN tokens and PERSON entities
nouns = [
    token.lemma_.lower()
    for token in doc
    if token.pos_ == "NOUN"
]

persons = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]

print("number of NOUN tokens:", len(nouns))
print("number of PERSON entities:", len(persons))
print("\nSample NOUNs:", nouns[:15])
print("Sample PERSONs:", persons[:10])

number of NOUN tokens: 153
number of PERSON entities: 2

Sample NOUNs: ['earth', 'stock', 'market', 'reason', 'target', 'air', 'traffic', 'problem', 'holiday', 'travel', 'world', 'man', 'ceo', 'company', 'official']
Sample PERSONs: ['nasa', 'copp clark limited cable']


In [55]:
# 8
from collections import Counter

noun_counts = Counter(nouns)
person_counts = Counter(persons)
top_nouns = noun_counts.most_common(20)
top_persons = person_counts.most_common(20)
print("Top 20 frequent NOUNs:🧾\n")
for word, cnt in top_nouns:
    print(f"{word:20} {cnt}")

print("\nTop 20 frequent PERSONs:👩\n")
for name, cnt in top_persons:
    print(f"{name:30} {cnt}")

Top 20 frequent NOUNs:🧾

market               6
stock                5
air                  3
traffic              3
holiday              3
index                3
news                 3
right                3
earth                2
reason               2
target               2
problem              2
travel               2
man                  2
company              2
state                2
safety               2
sex                  2
advice               2
decade               2

Top 20 frequent PERSONs:👩

nasa                           1
copp clark limited cable       1
